# Build Reusable Data Products




In [15]:
import os
import sys
from pathlib import Path
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark
from src.lake import GOLD, read_delta, write_delta

spark = create_spark("data-products")

# Enable dynamic partition overwrites for incremental slice updating
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

### Metadata & Helper functions 

In [16]:
def get_last_commit_timestamp(spark: SparkSession, delta_path: Path) -> datetime:
    if not os.path.exists(delta_path):
        return datetime.min
    try:
        delta_table = DeltaTable.forPath(spark, str(delta_path))
        latest_commit = delta_table.history(1).collect()[0]
        return latest_commit["timestamp"]
    except Exception:
        return datetime.min

def should_refresh(spark: SparkSession, target_product_path: Path, upstream_paths: list) -> bool:
    target_ts = get_last_commit_timestamp(spark, target_product_path)
    if target_ts == datetime.min:
        return True  # Product does not exist yet; must compute
      
    for src in upstream_paths:
        src_ts = get_last_commit_timestamp(spark, src)
        if src_ts > target_ts:
            return True  # Upstream source was updated after product generation
            
    return False

def with_metadata(df, schema_ver="1.0"):
    now = F.current_timestamp()
    return (
        df.withColumn("_data_source", F.lit("gold/integrated_taxi_trips"))
          .withColumn("_created_at", now)
          .withColumn("_refreshed_at", now)
          .withColumn("_schema_version", F.lit(schema_ver))
    )

def safe_write_delta(df, target_path: Path, partition_by: list = None, schema_ver: str = "1.0"):
    enriched_df = with_metadata(df, schema_ver=schema_ver)
    
    writer = enriched_df.write.format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true")
        
    if partition_by:
        writer = writer.partitionBy(*partition_by)
        
    writer.save(str(target_path))

In [17]:
integrated_path = GOLD / "integrated_taxi_trips"
integrated = read_delta(spark, integrated_path)
integrated.createOrReplaceTempView("integrated_taxi_trips")

# Evaluate schema evolution state on upstream integrated table
has_humidity = "humidity" in integrated.columns
has_aqi = "aqi" in integrated.columns
schema_version = "1.1" if (has_humidity or has_aqi) else "1.0"

print(f"--- Pipeline Execution Plan (Schema Version: {schema_version}) ---")

--- Pipeline Execution Plan (Schema Version: 1.1) ---


## Product 1: Daily Mobility Summary

In [18]:
prod_1_path = GOLD / "data_products" / "daily_borough_mobility"

if should_refresh(spark, prod_1_path, [integrated_path]):
    print("[REFRESHING] Product 1: daily_borough_mobility...")
    
    query_prod_1 = """
    SELECT
        pickup_date,
        pickup_borough,
        COUNT(*) AS total_trips,
        SUM(passenger_count) AS total_passengers,
        ROUND(SUM(total_amount), 2) AS total_revenue,
        ROUND(AVG(trip_distance), 2) AS avg_distance_miles,
        ROUND(AVG(fare_amount), 2) AS avg_fare
    FROM integrated_taxi_trips
    WHERE pickup_borough != 'UNKNOWN' AND pickup_date IS NOT NULL
    GROUP BY pickup_date, pickup_borough
    """
    df_prod_1 = spark.sql(query_prod_1)
    safe_write_delta(df_prod_1, prod_1_path, partition_by=["pickup_date"], schema_ver=schema_version)
    print("Product 1 refreshed successfully.")
else:
    print("[SKIPPED] Product 1: daily_borough_mobility (Up to date)")

[REFRESHING] Product 1: daily_borough_mobility...


Product 1 refreshed successfully.


## Product 2: Taxi Zone Monthly Demand

In [19]:
prod_2_path = GOLD / "data_products" / "taxi_zone_monthly_demand"

if should_refresh(spark, prod_2_path, [integrated_path]):
    print("[REFRESHING] Product 2: taxi_zone_monthly_demand...")
    
    query_prod_2 = """
    WITH monthly_zone_trips AS (
        SELECT
            TRUNC(pickup_date, 'MM') AS trip_month,
            pickup_location_id,
            pickup_borough,
            pickup_zone,
            pickup_date
        FROM integrated_taxi_trips
        WHERE pickup_zone != 'UNKNOWN' AND pickup_date IS NOT NULL
    )
    SELECT
        trip_month,
        pickup_location_id,
        pickup_borough,
        pickup_zone,
        COUNT(*) AS total_trips,
        COUNT(DISTINCT pickup_date) AS active_days,
        ROUND(CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT pickup_date), 2) AS avg_daily_trips
    FROM monthly_zone_trips
    GROUP BY trip_month, pickup_location_id, pickup_borough, pickup_zone
    """
    df_prod_2 = spark.sql(query_prod_2)
    safe_write_delta(df_prod_2, prod_2_path, partition_by=["trip_month"], schema_ver=schema_version)
    print("Product 2 refreshed successfully.")
else:
    print("[SKIPPED] Product 2: taxi_zone_monthly_demand (Up to date)")

[REFRESHING] Product 2: taxi_zone_monthly_demand...


Product 2 refreshed successfully.


## Product 3: Weather Impact Summary

In [20]:
prod_3_path = GOLD / "data_products" / "weather_impact_summary"

if should_refresh(spark, prod_3_path, [integrated_path]):
    print("[REFRESHING] Product 3: weather_impact_summary...")
    
    # Adapt query logic dynamically if 'humidity' schema evolution occurred
    humidity_select = "ROUND(AVG(humidity), 2) AS avg_humidity_pct," if has_humidity else ""
    
    query_prod_3 = f"""
    WITH binned_weather AS (
        SELECT
            trip_distance,
            {"humidity," if has_humidity else ""}
            CASE
                WHEN temperature_c IS NULL THEN 'Unknown'
                WHEN temperature_c < 0 THEN 'Freezing (<0°C)'
                WHEN temperature_c BETWEEN 0 AND 10 THEN 'Cold (0°C to 10°C)'
                WHEN temperature_c BETWEEN 10.01 AND 20 THEN 'Moderate (10°C to 20°C)'
                ELSE 'Warm (>20°C)'
            END AS temp_category,
            CASE
                WHEN wind_speed_ms IS NULL THEN 'Unknown'
                WHEN wind_speed_ms < 2 THEN 'Calm (<2 m/s)'
                WHEN wind_speed_ms BETWEEN 2 AND 6 THEN 'Moderate Wind (2-6 m/s)'
                ELSE 'High Wind (>6 m/s)'
            END AS wind_category
        FROM integrated_taxi_trips
        WHERE trip_distance > 0 AND trip_distance < 100
    )
    SELECT
        temp_category,
        wind_category,
        COUNT(*) AS trip_count,
        ROUND(AVG(trip_distance), 2) AS avg_distance_miles,
        {humidity_select}
        CURRENT_TIMESTAMP() AS _computed_at
    FROM binned_weather
    GROUP BY temp_category, wind_category
    ORDER BY temp_category, wind_category
    """
    df_prod_3 = spark.sql(query_prod_3)
    safe_write_delta(df_prod_3, prod_3_path, schema_ver=schema_version)
    print("Product 3 refreshed successfully.")
else:
    print("[SKIPPED] Product 3: weather_impact_summary (Up to date)")

[REFRESHING] Product 3: weather_impact_summary...


Product 3 refreshed successfully.


## Product 4: Air Quality Demand Summary

In [21]:
prod_4_path = GOLD / "data_products" / "air_quality_demand_summary"

if should_refresh(spark, prod_4_path, [integrated_path]):
    print("[REFRESHING] Product 4: air_quality_demand_summary...")
    
    aqi_select = "ROUND(AVG(aqi), 0) AS avg_aqi_index," if has_aqi else ""
    
    query_prod_4 = f"""
    WITH rounded_pm25 AS (
        SELECT
            ROUND(pm25, 0) AS pm25_level,
            {"aqi," if has_aqi else ""}
            pickup_date,
            pickup_hour
        FROM integrated_taxi_trips
        WHERE pm25 IS NOT NULL AND pickup_date IS NOT NULL AND pickup_hour IS NOT NULL
    )
    SELECT
        pm25_level,
        {aqi_select}
        COUNT(*) AS trips,
        COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS observed_hours,
        ROUND(CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT struct(pickup_date, pickup_hour)), 2) AS trips_per_hour
    FROM rounded_pm25
    GROUP BY pm25_level
    ORDER BY pm25_level DESC
    """
    df_prod_4 = spark.sql(query_prod_4)
    safe_write_delta(df_prod_4, prod_4_path, schema_ver=schema_version)
    print("Product 4 refreshed successfully.")
else:
    print("[SKIPPED] Product 4: air_quality_demand_summary (Up to date)")

[REFRESHING] Product 4: air_quality_demand_summary...


Product 4 refreshed successfully.


Product 5: Taxi Zone Weather Sensitivity Summary

In [22]:
prod_5_path = GOLD / "data_products" / "zone_weather_sensitivity"

if should_refresh(spark, prod_5_path, [integrated_path]):
    print("[REFRESHING] Product 5: zone_weather_sensitivity...")
    
    query_prod_5 = """
    WITH trips_with_weather AS (
        SELECT 
            pickup_zone,
            pickup_date,
            pickup_hour,
            NTILE(4) OVER (
                PARTITION BY pickup_zone
                ORDER BY (10 * sqrt(wind_speed_ms) - wind_speed_ms + 10.5) * (33 - temperature_c)
            ) AS weather_condition
        FROM integrated_taxi_trips
        WHERE pickup_zone IS NOT NULL AND pickup_date IS NOT NULL AND pickup_hour IS NOT NULL
    ),
    hourly_demand AS (
        SELECT 
            pickup_zone,
            weather_condition,
            COUNT(1) / COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS trips_per_hour
        FROM trips_with_weather
        GROUP BY pickup_zone, weather_condition
    ),
    pivoted AS (
        SELECT * FROM hourly_demand
        PIVOT (
            ROUND(AVG(trips_per_hour), 2)
            FOR weather_condition IN (1 AS coldest, 2 AS cool, 3 AS warm, 4 AS warmest)
        )
    )
    SELECT 
        pickup_zone,
        coldest, cool, warm, warmest,
        ROUND(((GREATEST(coldest, cool, warm, warmest) - LEAST(coldest, cool, warm, warmest)) / ((coldest + cool + warm + warmest) / 4.0)) * 100, 2) AS pct_variation
    FROM pivoted
    WHERE (coldest + cool + warm + warmest) / 4.0 >= 10
    ORDER BY pct_variation DESC
    """
    df_prod_5 = spark.sql(query_prod_5)
    safe_write_delta(df_prod_5, prod_5_path, schema_ver=schema_version)
    print("Product 5 refreshed successfully.")
else:
    print("[SKIPPED] Product 5: zone_weather_sensitivity (Up to date)")

[REFRESHING] Product 5: zone_weather_sensitivity...


Product 5 refreshed successfully.
